# Fun with JAX

This is a super quick illustration of the power of [JAX](https://github.com/google/jax), a Python library built by Google Research.

It should be run on a machine with a GPU --- for example, try Google Colab with the runtime environment set to include a GPU.

The aim is just to give a small taste of high performance computing in Python -- details will be covered later in the course.

We start with some imports

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

Let's check our hardware:

In [ ]:
!nvidia-smi

In [ ]:
!lscpu -e

## Transforming Data

A very common numerical task is to apply a transformation to a set of data points.

Our transformation will be the cosine function.

Here we evaluate the cosine function at 50 points.

In [ ]:
x = np.linspace(0, 10, 50)
y = np.cos(x)

Let's plot.

In [ ]:
fig, ax = plt.subplots()
ax.scatter(x, y)
plt.show()

Our aim is to evaluate the cosine function at many points.

In [ ]:
n = 50_000_000
x = np.linspace(0, 10, n)

### With NumPy

In [ ]:
%%time 

y = np.cos(x)

In [ ]:
%%time 

y = np.cos(x)

In [ ]:
x = None  

### With JAX

In [ ]:
x_jax = jnp.linspace(0, 10, n)

Let's time it.


JAX (via XLA) tries to run computations asynchronously:

It doesn’t block Python until results are needed.

This allows multiple operations to overlap and improves performance in pipelines.

But it also means that timing in a notebook can mislead you if you’re not careful.

When you call:

In [ ]:
%%time
    
y = jnp.cos(x_jax)
jax.block_until_ready(y); 

you’re telling JAX:

“Wait until all computations that produce `y` are fully finished.”

This synchronizes the host (Python) and device (GPU/CPU), ensuring that `%%time`
 captures the real, end-to-end execution time, including:
- Scheduling
- Compilation (if using jit)
- Actual GPU math

In [ ]:
%%time
    
y = jnp.cos(x_jax)
jax.block_until_ready(y); 

Here we change the input size --- can you explain why the timing changes?

In [ ]:
x_jax = jnp.linspace(0, 10, n + 1)

In [ ]:
%%time
    
y = jnp.cos(x_jax)
jax.block_until_ready(y);

In [ ]:
%%time
    
y = jnp.cos(x_jax)
jax.block_until_ready(y);

In [ ]:
x_jax = None  # Free memory

## Evaluating a more complicated function

In [ ]:
def f(x):
    y = np.cos(2 * x**2) + np.sqrt(np.abs(x)) + 2 * np.sin(x**4) - 0.1 * x**2
    return y

In [ ]:
fig, ax = plt.subplots()
x = np.linspace(0, 10, 100)
ax.plot(x, f(x))
ax.scatter(x, f(x))
plt.show()

Now let's try with a large array.

### With NumPy

In [ ]:
n = 50_000_000
x = np.linspace(0, 10, n)

In [ ]:
%%time 

y = f(x)

In [ ]:
%%time 

y = f(x)

### With JAX

In [ ]:
def f(x):
    y = jnp.cos(2 * x**2) + jnp.sqrt(jnp.abs(x)) + 2 * jnp.sin(x**4) - x**2
    return y

In [ ]:
x_jax = jnp.linspace(0, 10, n)

In [ ]:
%%time 

y = f(x_jax)
jax.block_until_ready(y);

In [ ]:
%%time 

y = f(x_jax)
jax.block_until_ready(y)

### Compiling the Whole Function

In [ ]:
f_jax = jax.jit(f)

In [ ]:
%%time 

y = f_jax(x_jax)
jax.block_until_ready(y);

In [ ]:
%%time 

y = f_jax(x_jax)
jax.block_until_ready(y);